# Тестирование "деревянных" моделей и бустингов.

In [ ]:
import polars as pl
from datetime import timedelta, date, datetime
from main_funcs import global_temporal_split, split_cold_start, ndcg_at_k, calculate_metrics, calculate_regression_metrics_vectorized

In [2]:
from catboost import CatBoostRanker, Pool

In [3]:
events_path = "../../data/dataset/small/marketplace/events"
items_path = "../../data/dataset/small/marketplace/items.pq"

In [4]:
events_df = pl.scan_parquet(events_path)
events_df.collect_schema()

Schema([('timestamp', Duration(time_unit='us')),
        ('user_id', UInt64),
        ('item_id', String),
        ('subdomain', String),
        ('action_type', String),
        ('os', String)])

In [5]:
events_df.head().collect(engine="streaming")

timestamp,user_id,item_id,subdomain,action_type,os
duration[μs],u64,str,str,str,str
1082d 89901µs,59328774,"""nfmcg_14777696""","""u2i""","""view""","""android"""
1082d 163483µs,66295302,"""nfmcg_26098539""","""catalog""","""view""","""android"""
1082d 864625µs,37542104,"""nfmcg_10840247""","""u2i""","""view""","""android"""
1082d 889192µs,35193548,"""nfmcg_8040572""","""catalog""","""view""","""android"""
1082d 936008µs,27256137,"""nfmcg_6770239""","""catalog""","""view""","""android"""


In [6]:
actions_count = events_df.group_by("action_type").agg(pl.len()).collect(engine="streaming")
actions_count.head()

action_type,len
str,u32
"""like""",41792
"""view""",126147783
"""clickout""",485448
"""click""",3772094


Классы явно не сбалансированны, просмотров гораздо больше. Чтоб сформировать таргет, нужно превратить частоту действия в числовой таргет так, чтобы редкие действия получали больший вес, а частые - меньший. Общее число наблюдений мы делим на число наблюдений этого конкретного действия.

Беру логарифмированный таргет, так как он показал наилучшие результаты в бейзлайне.

In [7]:
actions_count = actions_count.with_columns(
    (pl.col("len").sum() / pl.col("len")).log1p().alias("log_target") 
).drop("len")
actions_count

action_type,log_target
str,f64
"""like""",8.046339
"""view""",0.710044
"""clickout""",5.597366
"""click""",3.571844


In [8]:
events_df = events_df.join(actions_count.lazy(), on="action_type").with_columns([
    (pl.col("action_type") == "view").cast(pl.Int8).alias("target_view"),
    (pl.col("action_type") == "clickout").cast(pl.Int8).alias("target_clickout"),
    (pl.col("action_type") == "like").cast(pl.Int8).alias("target_like"),
    (pl.col("action_type") == "click").cast(pl.Int8).alias("target_click"),
])
events_df.head().collect(engine="streaming")

timestamp,user_id,item_id,subdomain,action_type,os,log_target,target_view,target_clickout,target_like,target_click
duration[μs],u64,str,str,str,str,f64,i8,i8,i8,i8
1082d 13h 39m 56s 787496µs,70080585,"""nfmcg_5994897""","""other""","""view""","""android""",0.710044,1,0,0,0
1082d 13h 39m 57s 117891µs,88415632,"""nfmcg_16103795""","""catalog""","""view""","""android""",0.710044,1,0,0,0
1082d 13h 39m 57s 123937µs,13995671,"""nfmcg_22860181""","""u2i""","""view""","""android""",0.710044,1,0,0,0
1082d 13h 39m 57s 256490µs,69588314,"""nfmcg_883048""","""u2i""","""view""","""android""",0.710044,1,0,0,0
1082d 13h 39m 57s 367567µs,49740670,"""nfmcg_24615706""","""catalog""","""view""","""android""",0.710044,1,0,0,0


In [9]:
null_info = events_df.null_count().collect()
print(null_info)

shape: (1, 11)
┌───────────┬─────────┬─────────┬───────────┬───┬────────────┬────────────┬────────────┬───────────┐
│ timestamp ┆ user_id ┆ item_id ┆ subdomain ┆ … ┆ target_vie ┆ target_cli ┆ target_lik ┆ target_cl │
│ ---       ┆ ---     ┆ ---     ┆ ---       ┆   ┆ w          ┆ ckout      ┆ e          ┆ ick       │
│ u32       ┆ u32     ┆ u32     ┆ u32       ┆   ┆ ---        ┆ ---        ┆ ---        ┆ ---       │
│           ┆         ┆         ┆           ┆   ┆ u32        ┆ u32        ┆ u32        ┆ u32       │
╞═══════════╪═════════╪═════════╪═══════════╪═══╪════════════╪════════════╪════════════╪═══════════╡
│ 0         ┆ 0       ┆ 0       ┆ 41792     ┆ … ┆ 0          ┆ 0          ┆ 0          ┆ 0         │
└───────────┴─────────┴─────────┴───────────┴───┴────────────┴────────────┴────────────┴───────────┘


Посмотрю по таблице items, какие есть пропуски.

In [10]:
ivents_lf = pl.scan_parquet(items_path)
ivents_lf.head().collect(engine="streaming")

item_id,brand_id,category,subcategory,price,embedding
str,u64,str,str,f64,"array[f32, 300]"
"""nfmcg_10000001""",137356,null,null,6.06364,"[-0.070853, 0.03246, … 0.082178]"
"""nfmcg_10000012""",53389,null,null,0.960436,"[-0.091099, 0.043168, … 0.060287]"
"""nfmcg_1000002""",34153,"""Fashion Accessories, Tech Add-…","""Hats, Scarves, and Shawls""",-1.793113,"[-0.056533, 0.082878, … 0.037475]"
"""nfmcg_10000034""",39543,null,null,3.416755,"[-0.112588, 0.043333, … -0.001615]"
"""nfmcg_10000039""",82320,"""Fashion Accessories, Tech Add-…","""Jewelry and Costume Jewelry""",4.293433,"[-0.157201, 0.026234, … 0.013904]"


In [11]:
null_info = ivents_lf.null_count().collect()
print("Количество строк:", ivents_lf.select(pl.len()).collect().item())
print("Количество колонок:", len(ivents_lf.columns))
print(null_info)

Количество строк: 2325409
Количество колонок: 6
shape: (1, 6)
┌─────────┬──────────┬──────────┬─────────────┬───────┬───────────┐
│ item_id ┆ brand_id ┆ category ┆ subcategory ┆ price ┆ embedding │
│ ---     ┆ ---      ┆ ---      ┆ ---         ┆ ---   ┆ ---       │
│ u32     ┆ u32      ┆ u32      ┆ u32         ┆ u32   ┆ u32       │
╞═════════╪══════════╪══════════╪═════════════╪═══════╪═══════════╡
│ 0       ┆ 0        ┆ 966395   ┆ 1233023     ┆ 2882  ┆ 73        │
└─────────┴──────────┴──────────┴─────────────┴───────┴───────────┘


/tmp/ipykernel_128468/2265000357.py:3: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  print("Количество колонок:", len(ivents_lf.columns))


Подготовлю данные для дальнейшей работы с моделями. Категориальные признаки оставляю, так как бустинги умеют с ними работать. Удаляю товары с пропусками в цене, так как их минимальное количество от общего числа. Эмбеддинги пока не беру, пропуски в категориях и подкатегориях заменяю на новую категорию "unknown". Строки в таблице действий с пропусками в поддомене также просто удаляю, так как их не много. Далее форматирую временную отметку в таблоице событий и собираю все в одну таблицу events_prepared, с которой и продолжу работу.

In [12]:
ACCEPTABLE_ACTIONS = ["view", "click", "clickout", "like"]

# Очистка items
items_lf = (
    ivents_lf
    .drop_nulls(subset=["price"])
    .with_columns([
        pl.col("category").fill_null("unknown").alias("category"),
        pl.col("subcategory").fill_null("unknown").alias("subcategory"),
        pl.col("brand_id").fill_null(-1).alias("brand_id"),
    ])
    .select([
        "item_id",
        "brand_id",
        "category",
        "subcategory",
        "price",
        # embedding пока не использую
    ])
)

# Подготовка events
MICROS_IN_DAY = 24 * 60 * 60 * 1_000_000

events_prepared = (
    pl.scan_parquet(events_path)
    .filter(pl.col("action_type").is_in(ACCEPTABLE_ACTIONS))
    .join(actions_count.lazy(), on="action_type", how="inner")
    .join(items_lf, on="item_id", how="inner")
    .with_columns([
        (pl.col("timestamp").cast(pl.Int64) // MICROS_IN_DAY).alias("day"),
    ])
    .filter(pl.col("subdomain").is_not_null())
    .select([
        "timestamp",
        "day",
        "user_id",
        "item_id",
        "subdomain",
        "action_type",
        "os",
        "log_target",
        "brand_id",
        "category",
        "subcategory",
        "price",
    ])
)

In [13]:
events_prepared.collect_schema()

Schema([('timestamp', Duration(time_unit='us')),
        ('day', Int64),
        ('user_id', UInt64),
        ('item_id', String),
        ('subdomain', String),
        ('action_type', String),
        ('os', String),
        ('log_target', Float64),
        ('brand_id', Int64),
        ('category', String),
        ('subcategory', String),
        ('price', Float64)])

In [14]:
events_prepared.head().collect(engine="streaming")

timestamp,day,user_id,item_id,subdomain,action_type,os,log_target,brand_id,category,subcategory,price
duration[μs],i64,u64,str,str,str,str,f64,i64,str,str,f64
1082d 14h 5m 54s 191173µs,1082,66994677,"""nfmcg_13757063""","""search""","""clickout""","""android""",5.597366,58573,"""Fashion Accessories, Tech Add-…","""Jewelry and Costume Jewelry""",1.0391
1082d 14h 11m 33s 690688µs,1082,74359703,"""nfmcg_15481893""","""u2i""","""clickout""","""android""",5.597366,175380,"""Footwear of All Types""","""Women's Footwear""",-0.312695
1082d 14h 29m 36s 170321µs,1082,1723761,"""nfmcg_13757063""","""u2i""","""clickout""","""ios""",5.597366,58573,"""Fashion Accessories, Tech Add-…","""Jewelry and Costume Jewelry""",1.0391
1082d 14h 4m 31s 346837µs,1082,24868371,"""nfmcg_13757063""","""other""","""view""","""android""",0.710044,58573,"""Fashion Accessories, Tech Add-…","""Jewelry and Costume Jewelry""",1.0391
1082d 14h 4m 32s 120111µs,1082,35526125,"""nfmcg_13757063""","""other""","""view""","""android""",0.710044,58573,"""Fashion Accessories, Tech Add-…","""Jewelry and Costume Jewelry""",1.0391


Беру данные только за 5% последних дней - максимум, который позволяет моя оперативная память (32гб).

In [17]:
#границы дней
raw_events = pl.scan_parquet(events_path)
MICROS_IN_DAY = 24 * 60 * 60 * 1_000_000

min_day, max_day = (
    raw_events
    .select(
        (pl.col("timestamp").cast(pl.Int64) // MICROS_IN_DAY).min().alias("min_day"),
        (pl.col("timestamp").cast(pl.Int64) // MICROS_IN_DAY).max().alias("max_day"),
    )
    .collect()
    .row(0)
)

days_all = (max_day - min_day) + 1
n_days_subset = int(days_all * 0.05)
if n_days_subset < 1:
    n_days_subset = 1

subset_min_day = max_day - n_days_subset + 1

events_subset = events_prepared.filter(pl.col("day") >= subset_min_day)

In [18]:
train, test = global_temporal_split(events_subset, 0.2)
print("Количество строк train:", train.select(pl.len()).collect().item())
print("Количество колонок train:", len(train.columns))
print("Количество строк test:", test.select(pl.len()).collect().item())
print("Количество колонок test:", len(test.columns))

Количество строк train: 4044137
Количество колонок train: 12


/tmp/ipykernel_128468/3991756773.py:3: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  print("Количество колонок train:", len(train.columns))


Количество строк test: 1072961
Количество колонок test: 12


/tmp/ipykernel_128468/3991756773.py:5: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  print("Количество колонок test:", len(test.columns))


In [19]:
# агрегируем события до уровня user-item в train и test
train_ui = (
    train
    .group_by(["user_id", "item_id"])
    .agg([
        pl.col("log_target").max().alias("log_target"),
        pl.col("day").max().alias("last_day"),       
        pl.col("price").mean().alias("price_mean"),
        pl.col("category").first().alias("category"),
        pl.col("subcategory").first().alias("subcategory"),
        pl.col("brand_id").first().alias("brand_id"),
        pl.col("os").first().alias("os"),
        pl.col("subdomain").first().alias("subdomain"),
        pl.col("action_type").first().alias("last_action_type"),
    ])
)

test_ui = (
    test
    .group_by(["user_id", "item_id"])
    .agg([
        pl.col("log_target").max().alias("log_target"),
        pl.col("day").max().alias("last_day"),
        pl.col("price").mean().alias("price_mean"),
        pl.col("category").first().alias("category"),
        pl.col("subcategory").first().alias("subcategory"),
        pl.col("brand_id").first().alias("brand_id"),
        pl.col("os").first().alias("os"),
        pl.col("subdomain").first().alias("subdomain"),
        pl.col("action_type").first().alias("last_action_type"),
    ])
)

In [20]:
train_ui.head().collect(engine="streaming")

user_id,item_id,log_target,last_day,price_mean,category,subcategory,brand_id,os,subdomain,last_action_type
u64,str,f64,i64,f64,str,str,i64,str,str,str
22404143,"""nfmcg_18407569""",0.710044,1298,0.727612,"""unknown""","""unknown""",140560,"""android""","""u2i""","""view"""
72815466,"""nfmcg_6168268""",0.710044,1298,2.942685,"""unknown""","""unknown""",117885,"""android""","""i2i""","""view"""
26656462,"""nfmcg_24541872""",0.710044,1298,0.641024,"""Electronic Devices and Gadgets""","""Portable Audio Electronics""",38547,"""android""","""u2i""","""view"""
22128483,"""nfmcg_25192318""",0.710044,1298,3.903882,"""Fashion Accessories, Tech Add-…","""Jewelry and Costume Jewelry""",137311,"""android""","""u2i""","""view"""
39043941,"""nfmcg_25192318""",0.710044,1298,3.903882,"""Fashion Accessories, Tech Add-…","""Jewelry and Costume Jewelry""",137311,"""android""","""u2i""","""view"""


In [21]:
test_ui.head().collect(engine="streaming")

user_id,item_id,log_target,last_day,price_mean,category,subcategory,brand_id,os,subdomain,last_action_type
u64,str,f64,i64,f64,str,str,i64,str,str,str
13279865,"""nfmcg_24198435""",0.710044,1307,-1.519781,"""Home Improvement and Countrysi…","""Fabric Products and Textiles""",219578,"""other""","""search""","""view"""
69446307,"""nfmcg_18106422""",0.710044,1307,1.655624,"""Miscellaneous Goods (Uncategor…","""unknown""",137311,"""android""","""u2i""","""view"""
78299771,"""nfmcg_6406311""",0.710044,1307,-0.726488,"""unknown""","""unknown""",189615,"""other""","""catalog""","""view"""
11262933,"""nfmcg_13861239""",0.710044,1307,0.728012,"""Household Electrical Appliance…","""Home Care and Health Appliance…",211031,"""other""","""search""","""view"""
75052295,"""nfmcg_27196039""",0.710044,1307,-1.774444,"""unknown""","""unknown""",141256,"""android""","""other""","""view"""


Заведу еще валидационную выборку.

In [22]:
min_day, max_day = (
    train_ui
    .select(
        pl.col("last_day").min().alias("min_day"),
        pl.col("last_day").max().alias("max_day"),
    )
    .collect()
    .row(0)
)

print(min_day, max_day)

1298 1305


In [23]:
days_all = (max_day - min_day) + 1
val_days = max(int(days_all * 0.1), 1)

val_min_day = max_day - val_days + 1
#делю трейн на трейн и валидацию в соотношении 9 к 1
train_train_ui = train_ui.filter(pl.col("last_day") < val_min_day)
valid_ui = train_ui.filter(pl.col("last_day") >= val_min_day)

Обучаю первую модель- вариант CatBoost для задач ранжирования CatBoostRanker с дефолтной функцией потерь для ранжирования YetiRank, беру ее так как она хорошо работает с категориальными признаками и не требует тонких настроек гиперпараметров, можно сразу увидеть результат для сравнения с нашим бейзлайном - SVD. 

In [25]:
train_train_df = train_train_ui.collect().to_pandas()
valid_df = valid_ui.collect().to_pandas()
test_df = test_ui.collect().to_pandas()

for df in (train_train_df, valid_df, test_df):
    df.rename(columns={"log_target": "target"}, inplace=True)

target_col = "target"
group_col = "user_id"

feature_cols = [
    "user_id",
    "item_id",
    "last_day",
    "price_mean",
    "category",
    "subcategory",
    "brand_id",
    "os",
    "subdomain",
    "last_action_type",
]

cat_features = [
    "user_id",
    "item_id",
    "category",
    "subcategory",
    "os",
    "subdomain",
    "last_action_type",
]

In [27]:
sort_cols = ["user_id", "last_day", "item_id"]

train_train_df = train_train_df.sort_values(sort_cols).reset_index(drop=True)
valid_df = valid_df.sort_values(sort_cols).reset_index(drop=True)
test_df = test_df.sort_values(sort_cols).reset_index(drop=True)

train_pool = Pool(
    data=train_train_df[feature_cols],
    label=train_train_df[target_col],
    group_id=train_train_df[group_col],
    cat_features=cat_features,
)

valid_pool = Pool(
    data=valid_df[feature_cols],
    label=valid_df[target_col],
    group_id=valid_df[group_col],
    cat_features=cat_features,
)

test_pool = Pool(
    data=test_df[feature_cols],
    label=test_df[target_col],
    group_id=test_df[group_col],
    cat_features=cat_features,
)

In [28]:
model = CatBoostRanker(
    loss_function="YetiRank",
    eval_metric="NDCG:top=15",         
    iterations=500,
    learning_rate=0.05,
    depth=8,
    random_seed=42,
    verbose=50,
    early_stopping_rounds=50,
    task_type="CPU"                   
)

model.fit(
    train_pool,
    eval_set=valid_pool,
    use_best_model=True,
)

Groupwise loss function. OneHotMaxSize set to 10
0:	test: 0.9920458	best: 0.9920458 (0)	total: 1.34s	remaining: 11m 8s
50:	test: 0.9936804	best: 0.9936804 (50)	total: 48.5s	remaining: 7m 6s
100:	test: 0.9938778	best: 0.9938778 (100)	total: 1m 24s	remaining: 5m 34s
150:	test: 0.9940612	best: 0.9940612 (150)	total: 2m 3s	remaining: 4m 45s
200:	test: 0.9941375	best: 0.9941420 (186)	total: 2m 45s	remaining: 4m 6s
250:	test: 0.9941748	best: 0.9941748 (250)	total: 3m 34s	remaining: 3m 32s
300:	test: 0.9942535	best: 0.9942586 (293)	total: 4m 28s	remaining: 2m 57s
350:	test: 0.9942479	best: 0.9942607 (326)	total: 5m 25s	remaining: 2m 18s
400:	test: 0.9942492	best: 0.9942739 (374)	total: 6m 23s	remaining: 1m 34s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.9942738558
bestIteration = 374

Shrink model to first 375 iterations.


CatBoostRanker(depth=8, early_stopping_rounds=50, eval_metric='NDCG:top=15', iterations=500, learning_rate=0.05, loss_function='YetiRank', random_seed=42, task_type='CPU', verbose=50)

In [29]:
test_df["pred_score"] = model.predict(test_pool)

In [ ]:
test_pl = pl.from_pandas(test_df)

# готовлю поларс датафрейм для передачи в функции подсчета метрик
user_based_df = (
    test_pl
    .group_by("user_id")
    .agg([
        pl.col("item_id").alias("true_items"),        # товары, с которыми юзер взаимодействовал
        pl.col("target").alias("relevancy"),          # их релевантности (log_target)
        pl.col("item_id").alias("predicted_items"),   
        pl.col("pred_score").alias("predicted_scores")
    ])
)

In [ ]:
ndcg_df = ndcg_at_k(
    user_based_df=user_based_df,
    relevancy_col="relevancy",
    true_items_col="true_items",
    predicted_items_col="predicted_items",
    predicted_score_col="predicted_scores",
    top_k=15,
)

mean_ndcg = ndcg_df["ndcg"].mean()
print("Test mean NDCG@15:", mean_ndcg)

Test mean NDCG@15: 0.9968857736538025


In [32]:
precision_15, recall_15 = calculate_metrics(
    df=user_based_df.select(["true_items", "predicted_items"]),
    k=15,
)

print("Test Precision@15:", precision_15)
print("Test Recall@15:", recall_15)

Test Precision@15: 0.33765755928501917
Test Recall@15: 0.9512438741450171
